In [20]:
from data_processing import load_and_prepare_dataset

df = load_and_prepare_dataset(
    'dataset.csv',
    drop_cols=[
        'fast_break_pct_avg_diff',
        'fast_break_pct_rolling_5_diff',
        'points_off_turnover_pct_avg_diff',
        'points_off_turnover_pct_rolling_5_diff'
    ]
)



In [21]:
import numpy as np

# Correlations with 'team_winner' (exclude 'spread', 'team_winner')
cols_without_spread = [col for col in df.columns if col not in ['spread', 'team_winner']]
corrs_team_winner = []

for col in cols_without_spread:
    if df[col].dtype.kind in 'bifc' and col != 'team_winner':
        try:
            corr = df[col].corr(df['team_winner'])
            if np.isfinite(corr):
                corrs_team_winner.append((col, corr))
        except Exception as e:
            continue

# Sort by absolute value of correlation, descending
corrs_team_winner = sorted(corrs_team_winner, key=lambda x: abs(x[1]), reverse=True)

print("Top 10 most correlated features with 'team_winner':")
for col, corr in corrs_team_winner[:10]:
    print(f"{col}: {corr:.4f}")

print("\n")

# Correlations with 'spread' (exclude 'team_winner', 'spread')
cols_without_team_winner = [col for col in df.columns if col not in ['team_winner', 'spread']]
corrs_spread = []

for col in cols_without_team_winner:
    if df[col].dtype.kind in 'bifc' and col != 'spread':
        try:
            corr = df[col].corr(df['spread'])
            if np.isfinite(corr):
                corrs_spread.append((col, corr))
        except Exception as e:
            continue

corrs_spread = sorted(corrs_spread, key=lambda x: abs(x[1]), reverse=True)

print("Top 10 most correlated features with 'spread':")
for col, corr in corrs_spread[:10]:
    print(f"{col}: {corr:.4f}")


Top 10 most correlated features with 'team_winner':
elo_diff: 0.4530
point_differential_avg_diff: 0.4092
power_rating_diff: 0.3801
win_loss_pct_diff: 0.3740
net_eff_avg_diff: 0.3739
margin_estimate: 0.3731
point_differential_ewm_diff: 0.3730
last_10_efficiency_diff: 0.3623
net_eff_ewm_diff: 0.3338
rank_diff: -0.3094


Top 10 most correlated features with 'spread':
elo_diff: 0.5605
point_differential_avg_diff: 0.5277
power_rating_diff: 0.4886
net_eff_avg_diff: 0.4823
margin_estimate: 0.4820
point_differential_ewm_diff: 0.4780
win_loss_pct_diff: 0.4663
last_10_efficiency_diff: 0.4649
net_eff_ewm_diff: 0.4266
non_conf_win_loss_pct_diff: 0.3969


In [ ]:
corr_before = df['point_differential_avg_diff'].corr(df['spread'])

best_corr = 0
best_shift = 0

# Try a reasonable range of values for the additive/subtractive shift
for shift in np.arange(0, 10.1, 0.1):
    temp = df['point_differential_avg_diff'].copy()
    temp[df['team_home_away'] == 1] += shift
    temp[df['team_home_away'] == 0] -= shift
    corr = temp.corr(df['spread'])
    if abs(corr) > abs(best_corr):
        best_corr = corr
        best_shift = shift

# Apply the best shift found to a new column
df['point_differential_avg_diff_adj'] = df['point_differential_avg_diff']
df.loc[df['team_home_away'] == 1, 'point_differential_avg_diff_adj'] = df['point_differential_avg_diff'] + best_shift
df.loc[df['team_home_away'] == 0, 'point_differential_avg_diff_adj'] = df['point_differential_avg_diff'] - best_shift
corr_after = df['point_differential_avg_diff_adj'].corr(df['spread'])
print(f"Best shift: {best_shift:.2f}")

print(corr_before, corr_after)

Best shift: 6.90
0.5276581270020926 0.5867922306652394


In [13]:
print("Number of zeros in each column (sorted descending):")
zero_counts = [(col, (df[col] == 0).sum()) for col in df.columns]
zero_counts_sorted = sorted(zero_counts, key=lambda x: x[1], reverse=True)
for col, zero_count in zero_counts_sorted:
    print(f"{col}: {zero_count}")


Number of zeros in each column (sorted descending):
team_winner: 111859
team_home_away: 109394
non_conf_win_loss_pct_diff: 100260
win_loss_pct_diff: 20856
tov_avg_diff: 18324
tov_ewm_diff: 18322
sos: 14739
sos_opp: 14739
quad_score: 12553
score_variance_diff: 12518
def_score_variance_diff: 12502
three_variance_diff: 12370
pace_variance_diff: 12320
off_eff_variance_diff: 12306
games_played: 7601
tov_vs_stl: 7054
stl_vs_tov: 7054
opponent_team_score_avg_diff: 6318
team_score_avg_diff: 6254
point_differential_avg_diff: 6186
block_rate_avg_diff: 6012
block_rate_ewm_diff: 6012
opponent_team_score_ewm_diff: 5892
three_pct_opponent_avg_diff: 5886
three_pct_opponent_ewm_diff: 5876
three_pct_avg_diff: 5874
team_score_ewm_diff: 5870
three_pct_ewm_diff: 5866
point_differential_ewm_diff: 5848
assist_to_fg_avg_diff: 5838
assist_to_fg_ewm_diff: 5836
drb_avg_diff: 5830
threes_advantage: 5828
threes_disadvantage: 5828
drb_ewm_diff: 5826
pace_diff: 5824
poss_avg_diff: 5824
orb_avg_diff: 5824
orb_ewm_di